# Phase 22 — causal and out-of-sample validation

All selection rules in this notebook are written before model outputs are inspected. The history and symbols have already been explored, so these results remain **exploratory chronological OOS evidence**, not a pristine final test.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

from dc_multiscale_research import (
    FeatureTier,
    ModelFamily,
    ObservationClock,
    REGIME_ORDER,
    add_future_diagnostics,
    build_multiscale_state,
    candidate_summary,
    dwell_lengths,
    eta_squared,
    filtering_prefix_invariant,
    fit_candidate,
    load_close,
    prepare_data,
    ratio_ladder,
)

DATA_ROOT = Path(r"D:\data_dump\market_data\raw\FXPIG-Server\5")
SYMBOL = "GBPUSD.r"
THRESHOLDS = ratio_ladder(0.002, 1.5, 5)
CLOCK = ObservationClock.UNION_EVENT
FEATURE_TIER = FeatureTier.DYNAMIC
STATE_COUNTS = (3, 4, 5)
HMM_FAMILIES = (ModelFamily.NORMAL_HMM, ModelFamily.STUDENTT_HMM)
RANDOM_STATE = 17
STABILITY_SEEDS = (29, 47)
MAX_ITER = 50
N_RESTARTS = 3
MINIMUM_VALIDATION_OCCUPANCY = 0.02
FUTURE_HORIZONS = (6, 24, 96)

## Frozen selection rule

A candidate is eligible only when:

1. fitting and causal filtering complete without error;
2. the filtering prefix-invariance audit passes;
3. every validation state occupies at least 2% of observations.

Among eligible candidates, choose the highest validation log-likelihood per observation. State count, emission family, and random seed are never selected on test results. The test partition is opened only after selection.

In [2]:
prices = load_close(DATA_ROOT, SYMBOL)
_, global_state, _ = build_multiscale_state(prices, THRESHOLDS, include_scale_state=False)
prepared = prepare_data(global_state, CLOCK, FEATURE_TIER)
print(f"Train: {len(prepared.frames['train']):,}")
print(f"Validation: {len(prepared.frames['validation']):,}")
print(f"Test: {len(prepared.frames['test']):,}")

Train: 2,045
Validation: 557
Test: 522


In [3]:
fits = []
selection_records = []
for family in HMM_FAMILIES:
    for n_states in STATE_COUNTS:
        fit = fit_candidate(
            prepared, family, n_states, RANDOM_STATE, max_iter=MAX_ITER, n_restarts=N_RESTARTS
        )
        fits.append(fit)
        record = candidate_summary(fit, prepared)
        prefix_pass = filtering_prefix_invariant(fit, prepared)
        record["filtering_prefix_invariant"] = prefix_pass
        record["eligible"] = (
            fit.error is None
            and prefix_pass is True
            and record.get("minimum_state_occupancy", 0.0) >= MINIMUM_VALIDATION_OCCUPANCY
        )
        selection_records.append(record)

selection_table = pd.DataFrame.from_records(selection_records)
display(selection_table.set_index(["family", "n_states"]))
eligible_table = selection_table.loc[selection_table["eligible"]].copy()
if eligible_table.empty:
    raise RuntimeError("No candidate passed the frozen validation eligibility rules.")
selected_row = eligible_table.sort_values("validation_log_likelihood_per_row", ascending=False).iloc[0]
selected_fit = next(
    fit for fit in fits
    if fit.family.value == selected_row["family"] and fit.n_states == int(selected_row["n_states"])
)
print(f"Selected on validation only: {selected_fit.family.value}, K={selected_fit.n_states}")

clock     tier error  \
family       n_states                                    
normal_hmm   3         union_event_time  dynamic  None   
             4         union_event_time  dynamic  None   
             5         union_event_time  dynamic  None   
studentt_hmm 3         union_event_time  dynamic  None   
             4         union_event_time  dynamic  None   
             5         union_event_time  dynamic  None   

                       validation_log_likelihood_per_row  \
family       n_states                                      
normal_hmm   3                                -25.503625   
             4                                -23.770640   
             5                                -15.796295   
studentt_hmm 3                                -29.261737   
             4                                -18.936978   
             5                                -10.867945   

                       minimum_state_occupancy  switch_rate  median_dwell  \
family       n_states                                                       
normal_hmm   3                        0.308797     0.214029           4.0   
             4                        0.109515     0.329137           2.0   
             5                        0.109515     0.750000           1.0   
studentt_hmm 3                        0.308797     0.214029           4.0   
             4                        0.134650     0.570144           1.0   
             5                        0.129264     0.591727           1.0   

                       mean_max_probability  baseline_adjusted_rand  \
family       n_states                                                 
normal_hmm   3                     1.000000                0.466753   
             4                     0.997412                0.349851   
             5                     0.997405                0.784826   
studentt_hmm 3                     0.999987                0.466753   
             4                     1.000000                0.296693   
             5                     1.000000                0.482655   

                       baseline_nmi  baseline_mapped_agreement  \
family       n_states                                            
normal_hmm   3             0.568454                   0.651706   
             4             0.520404                   0.542190   
             5             0.879244                   0.791741   
studentt_hmm 3             0.568454                   0.651706   
             4             0.438379                   0.500898   
             5             0.603379                   0.630162   

                       filtering_prefix_invariant  eligible  
family       n_states                                        
normal_hmm   3                               True      True  
             4                               True      True  
             5                               True      True  
studentt_hmm 3                               True      True  
             4                               True      True  
             5                               True      True

Selected on validation only: studentt_hmm, K=5


## Restart stability

Adjusted Rand agreement is permutation-invariant. It tests whether the selected family/state count repeatedly partitions validation observations similarly, not whether its semantic names are correct.

In [4]:
stability_records = []
for seed in STABILITY_SEEDS:
    refit = fit_candidate(
        prepared,
        selected_fit.family,
        selected_fit.n_states,
        seed,
        max_iter=MAX_ITER,
        n_restarts=N_RESTARTS,
    )
    stability_records.append({
        "seed": seed,
        "error": refit.error,
        "validation_ari_vs_selected": (
            adjusted_rand_score(selected_fit.states["validation"], refit.states["validation"])
            if refit.error is None else np.nan
        ),
        "test_ari_vs_selected": (
            adjusted_rand_score(selected_fit.states["test"], refit.states["test"])
            if refit.error is None else np.nan
        ),
        "prefix_invariant": filtering_prefix_invariant(refit, prepared),
    })
display(pd.DataFrame.from_records(stability_records).set_index("seed"))

,error,validation_ari_vs_selected,test_ari_vs_selected,prefix_invariant
seed,,,,
29,None,0.369389,0.326707,True
47,None,0.369389,0.326707,True


## Open the chronological test partition

These metrics are reported after validation selection. They evaluate stability, persistence, confidence, and relationship to the direct baseline. No test metric is used to repair or reselect the model.

In [5]:
test_states = selected_fit.states["test"]
test_probabilities = selected_fit.probabilities["test"]
test_baseline = prepared.frames["test"]["structural_regime_code"].to_numpy(dtype=np.int64)
test_occupancy = np.bincount(test_states, minlength=selected_fit.n_states) / len(test_states)
test_dwell = dwell_lengths(test_states)
test_metrics = pd.Series({
    "log_likelihood_per_row": selected_fit.log_likelihoods["test"] / len(test_states),
    "minimum_state_occupancy": test_occupancy.min(),
    "switch_rate": np.mean(test_states[1:] != test_states[:-1]),
    "median_dwell": np.median(test_dwell),
    "p90_dwell": np.quantile(test_dwell, 0.90),
    "mean_max_filtered_probability": test_probabilities.max(axis=1).mean(),
    "baseline_adjusted_rand": adjusted_rand_score(test_baseline, selected_fit.mapped_regimes["test"]),
    "baseline_nmi": normalized_mutual_info_score(test_baseline, selected_fit.mapped_regimes["test"]),
    "baseline_mapped_agreement": np.mean(test_baseline == selected_fit.mapped_regimes["test"]),
})
display(test_metrics.to_frame("test_value"))

,test_value
log_likelihood_per_row,-13.221069
minimum_state_occupancy,0.107280
switch_rate,0.558541
median_dwell,1.000000
p90_dwell,4.000000
mean_max_filtered_probability,1.000000
baseline_adjusted_rand,0.438095
baseline_nmi,0.577159
baseline_mapped_agreement,0.590038


## Diagnostic usefulness beyond renaming the baseline

Forward activity and absolute movement are diagnostic outcomes, not trading targets. Eta-squared measures how much held-out outcome variation is separated by each contemporaneous regime partition. It does not establish causation, profitability, or direction.

In [6]:
diagnostic_state = add_future_diagnostics(global_state, FUTURE_HORIZONS)
test_frame = prepared.frames["test"].copy()
test_frame["model_state"] = test_states
test_frame["mapped_model_regime"] = selected_fit.mapped_regimes["test"]
diagnostic_columns = []
for horizon in FUTURE_HORIZONS:
    diagnostic_columns.extend([
        f"future_abs_return_{horizon}",
        f"future_realized_vol_{horizon}",
        f"future_dcc_count_{horizon}",
    ])
test_frame = test_frame.join(diagnostic_state[diagnostic_columns], how="left")
usefulness_records = []
for column in diagnostic_columns:
    usefulness_records.append({
        "diagnostic": column,
        "direct_baseline_eta_squared": eta_squared(
            test_frame[column], test_frame["structural_regime_code"]
        ),
        "model_state_eta_squared": eta_squared(test_frame[column], test_frame["model_state"]),
    })
usefulness = pd.DataFrame.from_records(usefulness_records).set_index("diagnostic")
usefulness["model_minus_baseline"] = (
    usefulness["model_state_eta_squared"] - usefulness["direct_baseline_eta_squared"]
)
display(usefulness.style.format("{:.4f}"))

,direct_baseline_eta_squared,model_state_eta_squared,model_minus_baseline
diagnostic,,,
future_abs_return_6,0.0055,0.0203,0.0148
future_realized_vol_6,0.0189,0.0211,0.0022
future_dcc_count_6,0.0088,0.0055,-0.0033
future_abs_return_24,0.0131,0.0132,0.0001
future_realized_vol_24,0.0126,0.0227,0.0100
future_dcc_count_24,0.0083,0.0051,-0.0032
future_abs_return_96,0.0064,0.0102,0.0038
future_realized_vol_96,0.0042,0.0078,0.0037
future_dcc_count_96,0.0040,0.0006,-0.0034


# Phase 22 completion gate

1. Selection uses validation likelihood and eligibility rules, never test results.
2. Train-only scaling and train-only semantic mapping are preserved.
3. Filtered prefix invariance passes.
4. No state collapses below the predeclared occupancy floor on validation.
5. Restart sensitivity is reported rather than hidden.
6. Test persistence is evaluated together with confidence and switching; stickiness alone is not success.
7. Diagnostic future separation is compared with the direct baseline.
8. No directional alpha or trading claim is made.
9. Because this history was previously explored, the result remains exploratory OOS evidence.